# ActiveStep — Nested LOPO Assessment on Colab

This notebook reproduces and extends the Daphnet-based FOG detection
assessment. It mirrors `scripts/run_assessment.sh` but runs entirely in
Colab (no conda needed).

**Runtime → Change runtime type → T4 GPU** is recommended.

Pipeline: clone → install → audit → cache → smoke → full assessment → report.

## 1. Clone the repository

In [ ]:
import os, sys, pathlib

REPO_DIR = '/content/ActiveStep'
REPO_URL = 'https://github.com/mathew1046/ActiveStep.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists — pulling latest...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')
!ls -la

### Alternative: mount Google Drive (for private repos or large artifacts)

If the repo is private or you prefer Drive, uncomment and run this cell instead of the one above.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # Copy repo zip from Drive, or clone using a personal access token:
# # REPO_URL = 'https://<TOKEN>@github.com/mathew1046/ActiveStep.git'
# # !git clone {REPO_URL} /content/ActiveStep
# import os, sys
# REPO_DIR = '/content/ActiveStep'
# os.chdir(REPO_DIR)
# sys.path.insert(0, REPO_DIR)

## 2. Install dependencies

Colab already ships TensorFlow, NumPy, SciPy, pandas, scikit-learn, and matplotlib. We only need `tensorflow-model-optimization`.

In [ ]:
!pip install -q tensorflow-model-optimization>=0.7

import tensorflow as tf, numpy as np, sklearn, scipy
print(f'TF {tf.__version__} | NumPy {np.__version__} | sklearn {sklearn.__version__} | scipy {scipy.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 3. Dataset audit

Inventories the Daphnet recordings: subjects, durations, FOG minutes, event counts, timestamp integrity.

In [ ]:
!python -m src.data --audit --out models/dataset_audit.json

import json
with open('models/dataset_audit.json') as f:
    audit = json.load(f)
print(json.dumps(audit, indent=2))

## 4. Build window cache

Creates streaming-safe 2-second windows at 100 Hz with discrete annotation mapping and recording-boundary preservation. Cached to `.cache/windows` (~250 MB).

In [ ]:
!python -m src.data --build-cache --cache-dir .cache/windows

## 5. Smoke test (1-epoch mechanics check)

Quick validation on subjects 1–3 with 1 epoch and 400 windows max. Confirms the pipeline runs end-to-end before committing to the full run.

In [ ]:
!python -m src.train_nested --tag baseline_v1_smoke --epochs 1 --subjects 1,2,3 --max-windows 400 --n-boot 100
!python -m src.report --summary models/nested/baseline_v1_smoke/summary.json --out models/nested/baseline_v1_smoke/assessment_report.md

## 6. Full nested LOPO assessment

Leave-one-participant-out over all 10 subjects, 3 candidates (`cnn_fi`, `fi_only`, `logistic_fi`), 15 epochs, 1000 bootstrap iterations. This is the main run — expect 15–40 minutes on a T4 GPU.

In [ ]:
!python -m src.train_nested --tag baseline_v1 --epochs 15 --n-boot 1000

## 7. Generate report

In [ ]:
!python -m src.report --summary models/nested/baseline_v1/summary.json --out models/nested/baseline_v1/assessment_report.md

with open('models/nested/baseline_v1/assessment_report.md') as f:
    print(f.read())

## 8. Display key results

In [ ]:
import json, pandas as pd

with open('models/nested/baseline_v1/summary.json') as f:
    summary = json.load(f)

rows = []
for cand_name, cand in summary.items():
    tol = cand.get('tol_0s', {})
    pooled = tol.get('pooled', {})
    boot = tol.get('bootstrap', {})
    sens_ci = boot.get('event_sensitivity_ci95', [float('nan'), float('nan')])
    fc_ci = boot.get('false_cues_per_hour_ci95', [float('nan'), float('nan')])
    rows.append({
        'Candidate': cand_name,
        'Sensitivity': pooled.get('event_sensitivity', float('nan')),
        'Sens 95% CI': f'[{sens_ci[0]:.3f}, {sens_ci[1]:.3f}]',
        'False cues/h': pooled.get('false_cue_starts_per_nonfog_hour', float('nan')),
        'FC 95% CI': f'[{fc_ci[0]:.3f}, {fc_ci[1]:.3f}]',
        'PR-AUC': cand.get('pooled_window_pr_auc_endpoint', float('nan')),
    })

df = pd.DataFrame(rows)
print('=== Pooled participant-independent results ===')
print(df.to_string(index=False))

## 9. Per-fold breakdown

In [ ]:
fold_rows = []
for cand_name, cand in summary.items():
    for fold in cand.get('folds', []):
        fold_rows.append({
            'Candidate': cand_name,
            'Fold': f"S{fold.get('subject', '?'):02d}",
            'Events': fold.get('n_events', 0),
            'Detected': fold.get('n_detected', 0),
            'Sensitivity': fold.get('event_sensitivity', 0),
            'False cues/h': fold.get('false_cue_starts_per_nonfog_hour', 0),
            'Mean delay (s)': fold.get('delay_mean_s', float('nan')),
        })

fold_df = pd.DataFrame(fold_rows)
print(fold_df.to_string(index=False))

## 10. Save results to Google Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r models/nested/baseline_v1 /content/drive/MyDrive/ActiveStep/
# print('Copied to Drive.')

---

## Next: model improvement experiments

Once the baseline reproduces, add new cells below to iterate. Use a new `--tag` for each experiment so results are preserved separately.

### Experiment A: FI activity gate
Add an absolute locomotor-band power floor in `src/features.py` so the FI ratio doesn't explode during standing.

```python
# Edit src/features.py, then run:
# !python -m src.train_nested --tag fi_gate_v1 --epochs 15 --n-boot 1000
# !python -m src.report --summary models/nested/fi_gate_v1/summary.json --out models/nested/fi_gate_v1/assessment_report.md
```

### Experiment B: Endpoint-focused labels
Change window target from "any freeze in window" to "freeze at window end" in `src/data.py` to reduce detection delay.

### Experiment C: Class-weight tuning
Reduce the class weight from ~12× to 3–5× in `src/train_nested.py` and observe cross-subject score distributions.

### Experiment D: Temporal post-processing
Add score smoothing, consecutive-positive trigger, refractory period, and recovery hysteresis in `src/detector.py`. Tune these only inside inner folds.